# Phương án A1 — COMPACT MI–RFE–SVM

**EXPERIMENTAL_NON_OFFICIAL**. Chỉ thay RF-RFE thành step=5, target=1; giữ MI và SVM k=1..100. Chỉ chạy đến selection, không final refit và không mở Test. Checkpoint sau mỗi stage/k, heartbeat định kỳ. Mặc định preflight; không dùng Run all để bật training.


**Cell 1 — Mount MyDrive**


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive')
assert DRIVE.is_dir()
print('DRIVE:', DRIVE)


**Cell 2 — Paths, config và ngân sách vận hành**
Giữ đúng ba đường dẫn. Chọn tên folder thật của dataset; Cell 4 sẽ đọc catalog manifest. Đường dẫn có tên 5h không đặt giới hạn runtime 5 giờ. Session budget tính từ lúc mỗi worker bắt đầu; dừng ở safe boundary gần giới hạn.


In [ ]:
if globals().get('JOB', {}).get('running'):
    raise RuntimeError('Job đang chạy; không đổi cấu hình trong lúc này')
import os, sys, json, subprocess, threading, uuid, re
from datetime import datetime, timezone
DATASET_ROOT = DRIVE / 'CICFLOWMETER_FEATURE_CSV'
OUTPUT_ROOT = DRIVE / 'experiment_results'
LOCAL_ROOT = Path('/content/mi_rfe_svm_5h')
SELECTED_CONFIGS = ['FW256_FS8']  # chọn tên folder từ catalog Cell 4; có thể chọn nhiều
RUN_ID = 'RUN_A1_01'  # run mới cần ID chưa tồn tại; resume phải giữ đúng ID cũ
RUNTIME = {'session_budget_seconds': 36000,  # 10h; 0 = không đặt deadline
           'stop_margin_seconds': 1800, 'heartbeat_seconds': 15}
BUDGET = {'min_available_ram_bytes': 2*1024**3, 'max_process_rss_bytes': 40*1024**3}
TAKEOVER = {}  # config -> 'PREVIOUS RUNTIME STOPPED <lease ID thực tế>'
ACKNOWLEDGE_LOCAL_STOP = False
EVIDENCE_ROOT = None  # mặc định dùng snapshot manifest CICFLOW gốc đã đóng gói
print('DATASET_ROOT:', DATASET_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('LOCAL_ROOT:', LOCAL_ROOT)
print('CPU sklearn SVC/RF; không giả định GPU tăng tốc. Cache SVC giữ nguyên 10000.')


**Cell 3 — Source, Git commit và dependencies**
Điền SHA40 sau khi bạn push source A1. Checkout cũ chỉ được dùng khi sạch và đúng SHA; không pull/reset. Nếu tự bật cài dependencies, sau cài thành công hãy restart session, chạy lại Cells 1–3 và tắt INSTALL_DEPENDENCIES.


In [ ]:
APPROVED_COMMIT = ''  # điền full SHA40 từ git rev-parse HEAD
INSTALL_DEPENDENCIES = False
if globals().get('JOB', {}).get('running'):
    raise RuntimeError('Worker đang chạy; không đổi source')
if not re.fullmatch(r'[0-9a-fA-F]{40}', APPROVED_COMMIT):
    raise ValueError('Điền APPROVED_COMMIT đủ 40 ký tự')
APPROVED_COMMIT = APPROVED_COMMIT.lower()
REPO = Path('/content/CICFLOW_A1_' + APPROVED_COMMIT[:12])
URL = 'https://github.com/kasum00/RFforNetwork.git'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--', URL, str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', APPROVED_COMMIT], check=True)
def git(*args):
    return subprocess.run(['git', '-C', str(REPO), *args], check=True, capture_output=True, text=True).stdout.strip()
assert git('rev-parse', 'HEAD') == APPROVED_COMMIT
assert not git('status', '--porcelain'), 'Checkout có thay đổi; không reset để ép chạy'
SOURCE_ROOT = REPO / 'code_CICFLOW'
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(SOURCE_ROOT/'phuong_anA/requirements-colab.txt')], check=True)
    raise RuntimeError('Cài xong theo yêu cầu. Restart session, chạy lại Cells 1–3 và đặt INSTALL_DEPENDENCIES=False.')
sys.dont_write_bytecode = True
sys.path.insert(0, str(SOURCE_ROOT)) if str(SOURCE_ROOT) not in sys.path else None
import phuong_anA.contract as a1_contract
import mi_rfe_svm_colab.common as a0_common
if (Path(a1_contract.__file__).resolve() != (SOURCE_ROOT/'phuong_anA/contract.py').resolve()
        or Path(a0_common.__file__).resolve() != (SOURCE_ROOT/'mi_rfe_svm_colab/common.py').resolve()):
    raise RuntimeError('Kernel đang giữ module từ checkout cũ; restart session')
from phuong_anA.contract import paths, read, confirmation, scientific, info, redact
from phuong_anA.controller import launch
from phuong_anA.dataset import discover
from mi_rfe_svm_colab.preflight import environment
TEST_EVIDENCE = {'status': 'NOT_RUN'}
print('SOURCE_ROOT:', SOURCE_ROOT, 'COMMIT:', APPROVED_COMMIT)
print(json.dumps(environment(), indent=2))


**Cell 4 — Dataset catalog, manifest/isolation và resource metadata**
Không mở Test, kể cả footer. Train/Validation được hash và kiểm tra footer; số dòng lấy từ manifest. Nếu thiếu sidecar hoặc snapshot không khớp, dừng và cung cấp đúng artifact gốc; không tạo manifest để ép PASS.


In [ ]:
CATALOG = discover(DATASET_ROOT)
print(json.dumps(CATALOG, indent=2))
assert SELECTED_CONFIGS and len(set(SELECTED_CONFIGS)) == len(SELECTED_CONFIGS)
assert set(SELECTED_CONFIGS) <= set(CATALOG['configs']), 'Chọn config có manifest hợp lệ trong catalog'
def request_for(config, mode='preflight'):
    return {'source': str(SOURCE_ROOT), 'dataset': str(DATASET_ROOT), 'output': str(OUTPUT_ROOT),
            'local_root': str(LOCAL_ROOT), 'config': config, 'run_id': RUN_ID, 'mode': mode,
            'git_commit': APPROVED_COMMIT, 'runtime': RUNTIME, 'budget': BUDGET,
            'unit_tests': TEST_EVIDENCE, 'evidence_root': EVIDENCE_ROOT,
            'takeover_confirmation': TAKEOVER.get(config), 'acknowledge_local_stop': ACKNOWLEDGE_LOCAL_STOP}
for config in SELECTED_CONFIGS:
    result = launch(request_for(config, 'metadata'))
    print(config, 'STATUS:', result['status'])
    print('REASONS:', result.get('reasons', result.get('message')))
    print('RESOURCE:', result.get('resource_gate'))
    print('UNRELEASED LEASE:', result.get('unreleased_lease'))
    print('OUTPUT:', paths(request_for(config))[1])
print(json.dumps(scientific(SOURCE_ROOT), indent=2))


**Cell 5 — Unit tests synthetic (người dùng chủ động chạy)**
Không fit SVC thật, không đọc production dataset. Tests tạo fixture nhỏ, so sánh RFE step5 với sklearn và kiểm tra checkpoint per-k. Chỉ tiếp tục khi native exit0, unittest OK và evidence PASS; không skip/xfail.


In [ ]:
RUN_SYNTHETIC_TESTS = True
if RUN_SYNTHETIC_TESTS:
    evidence_root = LOCAL_ROOT / 'phuong_anA/_unit_evidence'
    evidence_root.mkdir(parents=True, exist_ok=True)
    stamp = uuid.uuid4().hex
    evidence_file = evidence_root / (stamp + '.json')
    test_command = [sys.executable, '-B', '-m', 'phuong_anA.selftest', '--source', str(SOURCE_ROOT), '--evidence', str(evidence_file)]
    test_started = datetime.now(timezone.utc).isoformat()
    test_process = subprocess.Popen(test_command, cwd=str(SOURCE_ROOT), stdout=subprocess.PIPE,
                                    stderr=subprocess.STDOUT, text=True, bufsize=1)
    with (evidence_root / (stamp + '.log')).open('x') as log:
        for line in test_process.stdout:
            line = redact(line); log.write(line); log.flush(); print(line, end='', flush=True)
    test_exit = test_process.wait()
    (evidence_root / (stamp + '_native.json')).write_text(json.dumps({'command': test_command,
        'start': test_started, 'end': datetime.now(timezone.utc).isoformat(), 'native_exit_code': test_exit}, indent=2))
    TEST_EVIDENCE = read(evidence_file) if evidence_file.exists() else {'status': 'FAIL'}
    print('TEST EVIDENCE:', TEST_EVIDENCE)
    if test_exit or TEST_EVIDENCE.get('status') != 'PASS':
        raise RuntimeError('A1 tests chưa PASS; không chạy training')
else:
    print('Tests chưa được bật; full run vẫn bị chặn')


**Cell 6 — Preflight cuối, chưa fit**
Cần PASS cho từng config trước khi chạy Cell 7. Resource FAIL thì BLOCKED; không tự giảm dữ liệu, k, số cây hoặc cache_size. Checkpoint cũ chỉ dùng khi source/data/algorithm identity khớp.


In [ ]:
PREFLIGHT = {}
for config in SELECTED_CONFIGS:
    result = launch(request_for(config, 'preflight'))
    PREFLIGHT[config] = result
    print(config, 'STATUS:', result['status'], 'REASONS:', result.get('reasons', result.get('message')))
    print('LAST CHECKPOINT:', result.get('last_checkpoint'))
    print('UNRELEASED LEASE:', result.get('unreleased_lease'))
    print('CONFIRMATION:', result.get('confirmation'))
if any(r.get('status') != 'PASS' for r in PREFLIGHT.values()):
    raise RuntimeError('Có config BLOCKED/FAIL. Xử lý nguyên nhân rồi preflight lại.')


**Cell 7 — Chạy mới hoặc resume; chỉ đến selection**
Giữ MODE=preflight khi kiểm tra. Đặt run cho ID mới; resume cho đúng ID cũ. Queue chạy tuần tự trong background để bạn dùng Cell 8 xem tiến độ/dừng. Không chạy Cell 7 lần nữa khi JOB đang running. Không có final refit/Test trong notebook này.


In [ ]:
MODE = 'preflight'  # preflight | run | resume | verify
# Sau graceful stop, đặt ACKNOWLEDGE_LOCAL_STOP=True trước lần resume.
def start_queue(requests):
    global JOB, JOB_THREAD
    if globals().get('JOB', {}).get('running'):
        raise RuntimeError('Queue vẫn chạy; không tạo job trùng')
    requests = json.loads(json.dumps(requests))
    JOB = {'running': True, 'current_request': None, 'results': [], 'error': None, 'stop_queue': False}
    def work():
        try:
            for req in requests:
                if JOB['stop_queue']: break
                JOB['current_request'] = req
                result = launch(req)
                JOB['results'].append({'config': req['config'], **result})
                if result['native_exit_code'] or result.get('status') not in ('PASS', 'PAUSED'):
                    raise RuntimeError(str(result.get('reasons', result.get('message', result))))
                if result.get('status') == 'PAUSED': break
        except BaseException as exc:
            JOB['error'] = str(exc)
        finally:
            JOB['running'] = False
            JOB['current_request'] = None
            print('Queue ended. Review JOB error/results; no automatic retry or new run.', flush=True)
    JOB_THREAD = threading.Thread(target=work, daemon=True)
    JOB_THREAD.start()
if MODE in ('run', 'resume'):
    requests = []
    for config in SELECTED_CONFIGS:
        req = request_for(config, MODE)
        exact = confirmation(req)
        if input('Nhập đúng: ' + exact + '\n') != exact: raise RuntimeError('Confirmation differs')
        req['confirmation'] = exact
        requests.append(req)
    start_queue(requests)
elif MODE in ('preflight', 'verify'):
    for config in SELECTED_CONFIGS:
        result = launch(request_for(config, MODE))
        print(json.dumps({k: result[k] for k in ('status', 'message', 'reasons', 'phase_status', 'best_k', 'validation_accuracy', 'directory') if k in result}, indent=2))
else:
    raise ValueError('A1 không có mode final refit hoặc Test')


**Cell 8 — Status, graceful stop và kết quả Validation**
ACTION=status chỉ đọc tiến độ thực. ACTION=stop gửi cờ local, chờ worker lưu checkpoint và PAUSED. Không ngắt runtime để mong lưu fit đang dở. Sau khi selection complete, đổi MODE=verify ở Cell 7 để xác minh checksum; kết quả không có Test metrics.


In [ ]:
ACTION = 'status'  # status | stop
if ACTION == 'stop':
    active = globals().get('JOB', {})
    if not active.get('running') or not active.get('current_request'):
        raise RuntimeError('Không có worker hiện tại trong notebook này')
    local, remote = paths(active['current_request'])
    (local/'STOP_REQUESTED').write_text(datetime.now(timezone.utc).isoformat())
    JOB['stop_queue'] = True
    print('REQUESTED LOCALLY. Wait for checkpoint SAVED and PAUSED; no SVC mid-fit resume.')
elif ACTION != 'status':
    raise ValueError(ACTION)
print('JOB:', {k: v for k, v in globals().get('JOB', {}).items() if k not in ('results', 'current_request')})
print('FINISHED CONFIGS:', [{'config': r.get('config'), 'status': r.get('status'), 'phase_status': r.get('phase_status'),
                              'native_exit_code': r.get('native_exit_code')} for r in globals().get('JOB', {}).get('results', [])])
for config in SELECTED_CONFIGS:
    local, remote = paths(request_for(config))
    heartbeat = local/'heartbeat.json'
    if not heartbeat.exists(): heartbeat = remote/'heartbeat.json'
    print(config, 'HEARTBEAT:', read(heartbeat) if heartbeat.exists() else 'NOT_AVAILABLE')
    marker = remote/'results/SELECTION_COMPLETE.json'
    if marker.exists():
        selection = read(remote/'results/selection.json')
        print(config, 'SAVED VALIDATION SELECTION:', json.dumps(selection, indent=2))
        print('RESULTS:', remote/'results')
    else:
        print(config, 'SELECTION NOT COMPLETE')
print('EXPERIMENTAL_NON_OFFICIAL. Test reads/evaluations: 0. No final refit.')
